# Phase 1 학습 — Kaggle GPU

맥북(MPS)은 epoch당 약 19분이라 50 epoch에 16시간이 걸린다. Kaggle GPU로 학습만 옮긴다.

**시작 전 노트북 설정 3가지** (오른쪽 패널에서):

| 설정 | 값 | 이유 |
|---|---|---|
| Accelerator | **GPU T4 x2** 또는 P100 | 없으면 CPU로 돌아 매우 느리다 |
| Internet | **On** | `git clone` 에 필요 |
| Input | 업로드한 데이터셋 추가 | 학습 데이터 |

> Accelerator 와 Internet 은 **휴대폰 인증**을 해야 켜진다.
> Settings → Phone Verification 에서 한 번만 하면 된다.

자세한 준비 절차는 저장소의 `docs/KAGGLE_SETUP.md` 참고.


## 1. GPU 붙었는지 확인
`cuda True` 가 나와야 한다. False 면 오른쪽 Accelerator 설정을 다시 볼 것.


In [ ]:
import torch
print('torch  :', torch.__version__)
print('cuda   :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU    :', torch.cuda.get_device_name(0))
else:
    print('⚠️ GPU 미연결 — Accelerator 를 GPU 로 바꾸고 세션을 재시작할 것')


## 2. 코드 받기
저장소는 공개라 인증 없이 클론된다. Internet 이 On 이어야 한다.


In [ ]:
import os

os.chdir('/kaggle/working')
if not os.path.exists('Capstone_Stock_Price_Prediction'):
    !git clone -q https://github.com/sungjunpk/Capstone_Stock_Price_Prediction.git
os.chdir('/kaggle/working/Capstone_Stock_Price_Prediction')
print('작업 위치:', os.getcwd())


## 3. 학습 데이터 붙이기

오른쪽 **Input** 에 추가한 데이터셋이 `/kaggle/input/` 아래에 마운트된다.
아래 셀이 zip 을 찾아 알아서 풀어 준다 — 데이터셋 이름을 몰라도 된다.


In [ ]:
import glob, os, shutil, zipfile

zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
parqs = glob.glob('/kaggle/input/**/panel.parquet', recursive=True)

os.makedirs('data/processed', exist_ok=True)

if zips:
    print('zip 발견:', zips[0])
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall('.')
elif parqs:
    src = os.path.dirname(parqs[0])
    print('parquet 발견:', src)
    for f in glob.glob(f'{src}/*.parquet'):
        shutil.copy(f, 'data/processed/')
else:
    raise SystemExit('❌ 입력 데이터를 못 찾았다. 오른쪽 Add Input 으로 데이터셋을 붙였는지 확인할 것')

!ls -lh data/processed/


## 4. 배관 점검 (먼저 이걸 돌린다)

전체 학습은 오래 걸리므로, 소규모로 한 번 돌려 데이터·코드가 정상인지 먼저 본다.
1~2분이면 끝난다.


In [ ]:
!python scripts/train.py --smoke


## 5. 전체 학습

T4 16GB 기준 `--batch-size 512` 까지 무난하다. 메모리 부족(OOM)이 나면 256으로 낮춘다.
코드가 CUDA를 감지하면 혼합정밀(AMP)과 DataLoader 워커를 자동으로 켠다.


In [ ]:
!python scripts/train.py --batch-size 512 --lr 1e-3


## 6. 결과 확인

리포트에는 **VSN 피처 중요도**가 들어 있다 — 해석가능성 분석의 근거로 쓴다.


In [ ]:
import glob, json

latest = sorted(glob.glob('outputs/reports/*.json'))[-1]
r = json.load(open(latest))
print('리포트   :', latest)
print('best val :', round(r['best_val_loss'], 6), '| epoch', r['best_epoch'])
print('샘플 수  :', r['sizes'])
print()
print('피처 중요도 상위 10')
for k, v in list(r['feature_importance'].items())[:10]:
    print(f'  {k:16s} {v:.4f}')


## 7. 결과 내려받기

Kaggle 은 `/kaggle/working` 아래만 저장·다운로드된다.
아래 셀을 돌리면 오른쪽 **Output** 패널에 zip 이 생기고, 거기서 받으면 된다.

받은 파일은 로컬 저장소의 같은 경로(`outputs/`)에 풀어 둔다.


In [ ]:
import shutil, os

os.makedirs('/kaggle/working/download', exist_ok=True)
for p in ['outputs/checkpoints', 'outputs/reports']:
    if os.path.exists(p):
        shutil.copytree(p, f'/kaggle/working/download/{os.path.basename(p)}', dirs_exist_ok=True)

shutil.make_archive('/kaggle/working/phase1_result', 'zip', '/kaggle/working/download')
print('생성 완료 → 오른쪽 Output 패널에서 phase1_result.zip 다운로드')
!ls -lh /kaggle/working/phase1_result.zip


---
## 자주 겪는 문제

| 증상 | 원인과 해결 |
|---|---|
| `cuda False` | Accelerator 가 None. GPU 로 바꾸고 **세션 재시작**(Run → Restart) |
| `git clone` 이 멈춤 | Internet 이 Off. 켜고 세션 재시작 |
| Accelerator/Internet 을 못 켬 | 휴대폰 인증 미완료. Settings → Phone Verification |
| `입력 데이터를 못 찾았다` | 오른쪽 **Add Input** 으로 데이터셋을 안 붙였다 |
| CUDA out of memory | `--batch-size` 를 512 → 256 → 128 로 낮춘다 |
| 세션이 끊김 | 무료 GPU 는 세션 12시간, 주당 30시간. 남은 시간은 오른쪽 상단에 표시된다 |

## 참고

- **수집은 로컬에서만 한다.** 키움 API 는 등록된 IP 에서만 호출되고,
  Kaggle 은 IP 가 매번 바뀐다. 그래서 여기에는 `.env`(API 키)가 필요 없다 —
  학습 과정에 API 호출이 전혀 없다.
- 데이터를 새로 만들면 로컬에서 `python scripts/package_data.py` 를 다시 돌리고
  Kaggle 데이터셋에 **New Version** 으로 올리면 된다.
